# SpaNCy-Shift Stage 2 — safeguard ablation ladder

**Question:** on the same Stage 1 input, our Stage 2 ties Stage 1 → MMD-ResNet on kBET but keeps
biology intact (silhouette 0.360 vs 0.329; 0 vs 5 shape-distorted markers; ECAD untouched vs moved).
**Which of our design choices buys that?** Each arm removes one safeguard from the published config
(`w_adv=0.3`, α=0.6, 10 epochs):

| Arm | What is removed | Prior evidence |
|---|---|---|
| `full` | nothing (published config) | — |
| `alpha=1` | partial blend: full correction applied (same model as `full`, inference only) | α sweep: silhouette 0.311 at α=1 |
| `no-mask` | bimodal masking: ECAD etc. also enter the MMD loss | indirect (MMD-ResNet moves ECAD) |
| `no-NTX` | NT-Xent contrastive term (`w_contrast=0`) | single seed: CD20 compressed |
| `no-Huber` | Huber penalty on the correction (`w_recon=0`) | none |
| `all-off` | all four at once | should approach MMD-ResNet's damage |

Every training arm in a seed starts from the same init (`torch.manual_seed(seed)` before each
`train()`), so differences within a seed come from the removed safeguard.

**Metrics** (the biology axes are the question; kBET is reported but not the target):
- 1D shape (peak / var / IQR ratios vs raw; distorted = peak < 0.80 or var outside [0.80, 1.25]), all
  markers including bimodal, plus **mean |correction| on bimodal markers** (direct masking check)
- per-sample silhouette (3 cell types, 19 samples) — all seeds
- positive population (±5%) and kBET — seed 0 only (`POP_SEEDS`, `KBET_SEEDS`)

**To run:** Runtime → Run all (GPU). Upload **`spancy_shift_ladder.py`** when §0 asks (a copy of
`spancy_shift.py` with one extra flag, `mask_bimodal`). Runtime ≈ 15 trainings × ~8 min ≈ 2 h, plus
kBET on 6 layers. For a quick first pass set `SEEDS = [0]` in §3 (~45 min of training).
**Paste back §9** (plus any error).

## 0. Colab Setup

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload spancy_shift_ladder.py (requested only if missing).
from google.colab import files
import os

NEEDED = ['spancy_shift_ladder.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()

for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:24s} {os.path.getsize(f):>10,} bytes')

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from scipy.stats import iqr as _iqr
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift_ladder as ss
from spancy_shift_ladder import (
    load_adata, detect_bimodal_markers, find_best_sample_per_marker,
    positive_population_table,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load data, bimodal markers, reference samples

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: DDPM scores the full joint distribution — no special handling required.")
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)
uni_markers = [m for m in marker_names if m not in bimodal_names]
uni_idx = [marker_names.index(m) for m in uni_markers]
bim_idx = [marker_names.index(m) for m in bimodal_names]
print(f'{len(uni_markers)} unimodal markers; bimodal: {bimodal_names}')

## 3. Train the ladder

Per seed: `full` is trained first; `alpha=1` reuses that model at inference. The other arms are
separate trainings from the same init. Inference uses a fixed seed (the GNN is deterministic at
inference anyway).

In [ ]:
SEEDS = [0, 1, 2]     # [0] = quick pass (~45 min of training)
KBET_SEEDS = [0]      # kBET is slow and not the question here
POP_SEEDS = [0]       # positive population has been 11/20 for every arm so far
K_NEIGHBORS = 15

BASE = dict(
    n_epochs=10, lr=1e-3, warmup_epochs=2, n_per_batch=512, k_neighbors=K_NEIGHBORS,
    hybrid_alpha=0.6, temperature=0.07, w_recon=0.1, w_contrast=0.5, w_adv=0.3, w_mmd=1.0,
    mmd_samples=256, grl_max=1.0, mask_bimodal=True,
)
# arm -> (training overrides; None = reuse the 'full' model, inference alpha)
ARMS = {
    'full':     ({}, 0.6),
    'alpha=1':  (None, 1.0),
    'no-mask':  ({'mask_bimodal': False}, 0.6),
    'no-NTX':   ({'w_contrast': 0.0}, 0.6),
    'no-Huber': ({'w_recon': 0.0}, 0.6),
    'all-off':  ({'mask_bimodal': False, 'w_contrast': 0.0, 'w_recon': 0.0}, 1.0),
}
KEY = {'full': 'full', 'alpha=1': 'alpha1', 'no-mask': 'nomask', 'no-NTX': 'nontx',
       'no-Huber': 'nohuber', 'all-off': 'alloff'}


def lkey(arm, seed):
    return f'{KEY[arm]}_s{seed}'


adata_norm = None
t0 = time.time()
for seed in SEEDS:
    full_model = None
    for arm, (over, alpha) in ARMS.items():
        if over is None:
            m, sc, ref = full_model
        else:
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            m, sc, ref, _ = ss.train(
                adata, device_str=DEVICE,
                bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
                ref_sample_per_marker=ref_sample_per_marker,
                **{**BASE, **over},
            )
            if arm == 'full':
                full_model = (m, sc, ref)
        torch.manual_seed(100)
        torch.cuda.manual_seed_all(100)
        out = ss.normalize_adata(
            adata, m, sc, ref, hybrid_alpha=alpha, k_neighbors=K_NEIGHBORS,
            inference_batch_size=2048, device_str=DEVICE,
            layer_name='normalized', keep_base_layer=True,
        )
        g = np.asarray(out.layers['normalized'], dtype=np.float32)
        if adata_norm is None:
            adata_norm = out                  # keeps 'normalized_base' (Stage 1)
            del adata_norm.layers['normalized']
            X_base = np.asarray(adata_norm.layers['normalized_base'])
            base_log = np.log1p(np.clip(X_base, 0, None)).astype(np.float32)
        else:
            del out
        adata_norm.layers[lkey(arm, seed)] = g
        del g
        if over is not None and arm != 'full':
            del m, sc, ref
        gc.collect()
        print(f'  seed {seed}  {arm:9s} done  ({(time.time() - t0) / 60:.1f} min)')
    del full_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

LAYERS = [('normalized_base', 'Stage 1', None, None)]
LAYERS += [(lkey(a, s), f'{a} s{s}', a, s) for s in SEEDS for a in ARMS]
print(f'\n{len(LAYERS)} layers: {[l[1] for l in LAYERS]}')

## 4. 1D shape + correction on bimodal markers

Same per-(marker, sample) ratios as cell 6b of the other notebooks (`_shape_ratios` copied verbatim),
but computed for **all** markers so the bimodal ones (ECAD, …) are visible. `bim_delta` = mean
|log1p(arm) − log1p(Stage 1)| over bimodal markers: ~0 means bimodal markers were left alone.

In [ ]:
N_SHAPE_BINS = 80
MIN_CELLS = 50
PEAK_LO, VAR_LO, VAR_HI = 0.80, 0.80, 1.25

sample_ids_shape = adata_norm.obs['sample_id'].values
sample_idx = {s: np.where(sample_ids_shape == s)[0] for s in np.unique(sample_ids_shape)}
sample_idx = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


shape_rows, shape_sum = [], {}
for key, label, arm, seed in LAYERS:
    X_log = np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        ps, vs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(X_raw_log[idx, mi], X_log[idx, mi])
            if np.isfinite(p):  ps.append(p)
            if np.isfinite(v):  vs.append(v)
            if np.isfinite(ir): irs.append(ir)
        shape_rows.append(dict(key=key, marker=mname, peak=np.mean(ps), var=np.mean(vs), iqr=np.mean(irs)))
    df = pd.DataFrame([r for r in shape_rows if r['key'] == key]).set_index('marker')
    bad = df[(df.peak < PEAK_LO) | (df['var'] < VAR_LO) | (df['var'] > VAR_HI)]
    u = df.loc[uni_markers]
    shape_sum[key] = dict(
        peak=float(u.peak.mean()), var=float(u['var'].mean()), iqr=float(u.iqr.mean()),
        dist=len(bad), dist_list=', '.join(bad.index),
        ecad_var=float(df.loc['ECAD', 'var']) if 'ECAD' in df.index else np.nan,
        ecad_iqr=float(df.loc['ECAD', 'iqr']) if 'ECAD' in df.index else np.nan,
        bim_delta=float(np.abs(X_log[:, bim_idx] - base_log[:, bim_idx]).mean()) if bim_idx else 0.0,
        uni_delta=float(np.abs(X_log[:, uni_idx] - base_log[:, uni_idx]).mean()),
    )
    del X_log
    gc.collect()
shape_df = pd.DataFrame(shape_rows)

print('Shape summary (peak/var/iqr = unimodal means; dist = ALL markers incl. bimodal):')
print(pd.DataFrame(shape_sum).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).round(4).to_string())

## 5. Silhouette (all seeds)

In [ ]:
# Silhouette — UniFORM-style, 3 cell types, PER SAMPLE (same method + guard as the other notebooks).
N_SIL = 3000
MIN_TYPE_CELLS = 50

anchor_thresholds = {}
for mname in ['ECAD', 'CD45', 'aSMA']:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(X_log_preview[:, mi].reshape(-1, 1))
    anchor_thresholds[mname] = float(np.mean(gmm.means_))
print('Anchor GMM thresholds (log1p, raw):', {k: round(v, 3) for k, v in anchor_thresholds.items()})

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']] = 2
cell_type[X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']] = 1
cell_type[X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']] = 0

# Per-sample selection is decided once (deterministic guard, rng 42 per sample as before).
sample_ids_all = adata_norm.obs['sample_id'].values
sil_sel, dropped = {}, []
for sample in sorted(np.unique(sample_ids_all)):
    smask_idx = np.where(sample_ids_all == sample)[0]
    s_labels = cell_type[smask_idx]
    big = [c for c in (0, 1, 2) if int((s_labels == c).sum()) >= MIN_TYPE_CELLS]
    if len(big) < 2:
        dropped.append(sample)
        continue
    valid = np.where(np.isin(s_labels, big))[0]
    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sil_sel[sample] = (smask_idx[sel], s_labels[sel])

sil_rows = {}
for key in ['raw'] + [k for k, _, _, _ in LAYERS]:
    X = X_raw if key == 'raw' else np.asarray(adata_norm.layers[key])
    sil_rows[key] = {s: float(silhouette_score(np.log1p(np.clip(X[ii], 0, None)), lab, metric='euclidean'))
                     for s, (ii, lab) in sil_sel.items()}
sil_df = pd.DataFrame(sil_rows)
sil_df.loc['Mean'] = sil_df.mean()
print(f'{len(sil_sel)} samples passed the guard (dropped: {dropped})')
print(sil_df.loc[['Mean']].round(4).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).to_string())

## 6. Positive population (`POP_SEEDS`)

In [ ]:
pop_sum = {}
for key, label, arm, seed in LAYERS:
    if seed is not None and seed not in POP_SEEDS:
        continue
    pt = positive_population_table(adata_norm, norm_layer=key, sample_col='sample_id')
    s = pt.groupby('marker')['delta'].mean()
    pop_sum[key] = dict(pass_5pct=int((s.abs() < 5).sum()),
                        mean_abs_delta=float(pt['delta'].abs().mean()))
    print(f'  {label:16s} {pop_sum[key]["pass_5pct"]}/20 within +/-5%   '
          f'mean |delta| {pop_sum[key]["mean_abs_delta"]:.2f}%')

## 8. kBET (`KBET_SEEDS`)

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

# Stage 1 kBET is deterministic (0.6202) -> skipped.
ABL_LABEL = {k: l for k, l, _, _ in LAYERS}
EVAL_LAYERS = [k for k, _, _, s in LAYERS if s is not None and s in KBET_SEEDS]
print(f'kBET on {len(EVAL_LAYERS)} layers: {[ABL_LABEL[k] for k in EVAL_LAYERS]}')

In [ ]:
import anndata as ad
import gc

all_kbet = {}
umap_coords = {}

# MEMORY NOTE: the original cell did `adata_kbet = adata_norm.copy()`, which duplicates
# EVERY layer (~141 MB each; 6 layers once the ablation arms exist = ~850 MB per
# iteration, 5 iterations). pegasus only ever reads .X here, so build a minimal
# AnnData instead: X + obs + var. That keeps the per-iteration cost at one 141 MB
# array rather than ~850 MB, and it is faster too.
for layer_name in EVAL_LAYERS:
    label = ABL_LABEL.get(layer_name, layer_name)
    print(f'\n{"-"*55}\n  {label}  ({layer_name})\n{"-"*55}')

    adata_kbet = ad.AnnData(
        X=np.asarray(adata_norm.layers[layer_name], dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}
        finally:
            del parts, adata_g, mmdata

    all_kbet[layer_name] = layer_res
    del adata_kbet
    gc.collect()

print('\nDone.')

## 9. Ladder result (paste this)

Reference rows are hard-coded: Stage 1 → MMD-ResNet from `spancy_shift_mmdresnet_s1_wadv03.ipynb`
(3 seeds). A safeguard is flagged **MATTERS** if removing it lowers silhouette by > 0.005 at every
seed, or adds ≥ 1 shape-distorted marker on average, or more than doubles the correction on bimodal
markers (to > 0.02 log1p). The numbers are printed either way — the flag is only a guide.

In [ ]:
recs = []
for key, label, arm, seed in LAYERS:
    if arm is None:
        continue
    r = dict(arm=arm, seed=seed, sil=float(sil_df.loc['Mean', key]), **{
        k: v for k, v in shape_sum[key].items() if k != 'dist_list'})
    if key in all_kbet:
        kv = [x['kBET'] for x in all_kbet[key].values() if not np.isnan(x['kBET'])]
        r['kBET'] = float(np.mean(kv)) if kv else np.nan
    if key in pop_sum:
        r['pos_pop'] = pop_sum[key]['pass_5pct']
    recs.append(r)
res = pd.DataFrame(recs)

cols = ['sil', 'peak', 'var', 'iqr', 'dist', 'ecad_var', 'ecad_iqr', 'bim_delta', 'uni_delta']
agg = res.groupby('arm', sort=False)[cols].agg(['mean', 'std']).round(4)
s1 = shape_sum['normalized_base']
print('=' * 100)
print(f'MEAN +/- SD over seeds {SEEDS}  (unimodal peak/var/iqr; dist = all markers; '
      f'bim_delta / uni_delta = mean |correction| in log1p)')
print('=' * 100)
print(agg.to_string())
print(f'\nStage 1:  sil {float(sil_df.loc["Mean", "normalized_base"]):.4f}  '
      f'peak {s1["peak"]:.3f} var {s1["var"]:.3f} iqr {s1["iqr"]:.3f}  dist {s1["dist"]}  '
      f'ECAD var {s1["ecad_var"]:.3f} iqr {s1["ecad_iqr"]:.3f}   raw sil {float(sil_df.loc["Mean", "raw"]):.4f}')
print('S1 -> MMD-ResNet (3 seeds, w_adv03 notebook): kBET 0.7176 +/- 0.0235, sil 0.3288 +/- 0.0031,')
print('   5 distorted (CD3 var x2.3, ChromA x2.2, ECAD IQR x1.8), pos-pop 9/20')

print('\nSeed-0 extras (kBET, positive population):')
print(res[res.seed.isin(KBET_SEEDS + POP_SEEDS)][['arm', 'seed'] + [c for c in ['kBET', 'pos_pop'] if c in res]]
      .round(4).to_string(index=False))

print('\nDistorted markers per arm, seed 0:')
for key, label, arm, seed in LAYERS:
    if seed == SEEDS[0]:
        print(f'  {label:16s} {shape_sum[key]["dist"]}  {shape_sum[key]["dist_list"]}')

print('\n' + '=' * 100)
print('PER SAFEGUARD — removed vs full, paired within seed')
print('=' * 100)
full = res[res.arm == 'full'].set_index('seed')
for arm in [a for a in ARMS if a != 'full']:
    a = res[res.arm == arm].set_index('seed')
    dsil = a['sil'] - full['sil']
    ddist = float((a['dist'] - full['dist']).mean())
    bim_full, bim_arm = float(full['bim_delta'].mean()), float(a['bim_delta'].mean())
    dvar = float((a['var'] - full['var']).mean())
    flags = []
    if (dsil < -0.005).all():
        flags.append('silhouette')
    if ddist >= 1:
        flags.append('shape')
    if bim_arm > 0.02 and bim_arm > 2 * bim_full:
        flags.append('bimodal markers moved')
    verdict = 'MATTERS (' + ', '.join(flags) + ')' if flags else 'no measurable biology effect'
    print(f'  {arm:9s} sil ' + ' '.join(f's{s} {v:+.4f}' for s, v in dsil.items())
          + f' | dist {ddist:+.1f} | uni var {dvar:+.3f} | bim_delta {bim_full:.3f} -> {bim_arm:.3f}'
          + f'\n            -> {verdict}')